## YOLO11m + ROI Segmentation Refinement

**Objective:**
Two-stage wound pipeline with structured segmentation experiments.

Stage 1: YOLO11m-seg detects wound bounding boxes + coarse masks.

Stage 2: A configurable ROI segmentation model (`unetplusplus` baseline, optional `deeplabv3plus`) refines masks on cropped ROIs.

This notebook now supports:
- GT-only ROI training
- mixed GT / jitter / cached-YOLO ROI training
- resolution sweeps (256 / 384 / 512)
- boundary-aware loss
- multi-scale combined refinement

**Dataset:**
`data/wound_focus_clean/` with pre-built train / val / test splits.


## 1: Setup / Imports / Configuration

In [ ]:
%matplotlib inline

import json
import math
import sys
import time
import platform
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml

SCRIPT_DIR = Path.cwd().resolve()
# Notebook-only training: cwd must be experiments/YOLO11m_UNetPP
if not (SCRIPT_DIR / "config.yaml").is_file() or not (SCRIPT_DIR / "train_model.py").is_file():
    raise RuntimeError(
        "Kernel working directory must be experiments/YOLO11m_UNetPP "
        f"(config.yaml + train_model.py not found in {SCRIPT_DIR})"
    )
PROJECT_ROOT = SCRIPT_DIR.parent.parent
sys.path.insert(0, str(SCRIPT_DIR))

from pipeline_utils import (
    set_seed,
    get_device,
    load_config,
    prepare_yolo_dataset,
    validate_yolo_dataset,
    create_unet_datasets,
    make_unet_dataloaders,
    unet_collate_fn,
    WoundDataset,
    IMAGENET_MEAN,
    IMAGENET_STD,
    WOUND_ONLY_CLASSES,
)
from train_model import (
    build_yolo_model,
    train_yolo,
    evaluate_yolo,
    predict_yolo,
    build_segmentation_model,
    build_unet_model,
    train_unet,
    build_unet_criterion,
    train_one_epoch_unet,
    validate_one_epoch_unet,
    evaluate_unet_metrics,
    save_unet_checkpoint,
    load_unet_checkpoint,
    save_unet_training_curves,
    combined_inference,
    evaluate_combined,
    calculate_wound_area,
    predict_single_image,
    generate_report,
    save_global_metrics_summary,
    display_results_predictions,
)

print("=" * 60)
print("Libraries imported successfully")
print("=" * 60)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
print("=" * 60)

# Load configuration (validate combined: section against CombinedInferenceConfig)
CONFIG = load_config(SCRIPT_DIR / "config.yaml", validate_combined=True)
set_seed(CONFIG.get("seed", 42))
device = get_device()

# Resolve paths
output_dir = SCRIPT_DIR / "checkpoints"
results_dir = SCRIPT_DIR / "results"
reports_dir = SCRIPT_DIR / "reports"
for d in [output_dir, results_dir, reports_dir]:
    d.mkdir(parents=True, exist_ok=True)

print(f"\nDevice: {device}")
print(f"Experiment name: {CONFIG.get('experiment_name')}")
print(f"\nConfiguration:")
for section in ["yolo", "unet", "combined"]:
    print(f"\n  [{section.upper()}]")
    for k, v in CONFIG.get(section, {}).items():
        print(f"    {k}: {v}")

_unet = CONFIG.get("unet", {})
_comb = CONFIG.get("combined", {})
print("\n[Segmentation experiment lock]")
print(
    f"  architecture={_unet.get('architecture')}, "
    f"input_size={_unet.get('input_size')}, "
    f"roi_crop_mode={_unet.get('roi_crop_mode')}, "
    f"loss_type={_unet.get('loss_type')}, "
    f"resume_checkpoint={_unet.get('resume_checkpoint')}"
)
print("\n[Combined pipeline lock]")
print(
    f"  yolo_conf_thresh={_comb.get('yolo_conf_thresh')}, "
    f"unet_mask_thresh={_comb.get('unet_mask_thresh')}, "
    f"roi_padding={_comb.get('roi_padding')}, "
    f"multi_scale_refinement={_comb.get('multi_scale_refinement')}, "
    f"multi_scale_fusion={_comb.get('multi_scale_fusion')}, "
    f"refinement_postprocess={_comb.get('refinement_postprocess')}"
)

## 2: Dataset Validation and Loading

In [ ]:
# ============================================================================
# 2.1 Convert COCO annotations to YOLO segmentation format
# ============================================================================

print("Converting COCO -> YOLO label format...")
dataset_yaml = prepare_yolo_dataset(CONFIG, SCRIPT_DIR)
print(f"\nDataset YAML: {dataset_yaml}")

print("\nValidating YOLO dataset...")
is_valid = validate_yolo_dataset(dataset_yaml)
if not is_valid:
    raise RuntimeError("YOLO dataset validation FAILED.")
print()

# ============================================================================
# 2.2 Create U-Net++ ROI datasets and dataloaders
# ============================================================================

print("Creating U-Net++ ROI datasets...")
unet_train_ds, unet_val_ds, unet_test_ds = create_unet_datasets(CONFIG, SCRIPT_DIR)
print(f"  ROI samples — Train: {len(unet_train_ds)}, Val: {len(unet_val_ds)}, Test: {len(unet_test_ds)}")

unet_cfg = CONFIG["unet"]
unet_train_loader, unet_val_loader = make_unet_dataloaders(
    unet_train_ds, unet_val_ds,
    batch_size=unet_cfg.get("batch_size", 16),
    num_workers=CONFIG.get("num_workers", 0),
)
unet_test_loader = torch.utils.data.DataLoader(
    unet_test_ds,
    batch_size=unet_cfg.get("batch_size", 16),
    shuffle=False,
    num_workers=CONFIG.get("num_workers", 0),
    pin_memory=torch.cuda.is_available(),
    collate_fn=unet_collate_fn,
)

# ============================================================================
# 2.3 Create evaluation dataset (full images for combined pipeline)
# ============================================================================

data_root = (PROJECT_ROOT / CONFIG["data_root"]).resolve()
test_ann = (PROJECT_ROOT / CONFIG["ann_test"]).resolve()
eval_dataset = WoundDataset(
    root=str(data_root),
    annotation_file=str(test_ann),
    image_size=(CONFIG["yolo"].get("image_size", 640), CONFIG["yolo"].get("image_size", 640)),
)
print(f"  Full-image test set: {len(eval_dataset)} images")

## 3: Sample Visualization and Model Initialization

In [ ]:
# ============================================================================
# 3.1 Visualize U-Net++ ROI training samples with GT masks
# ============================================================================

NUM_VIS = 4
fig, axes = plt.subplots(2, NUM_VIS, figsize=(4 * NUM_VIS, 8))

mean_t = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std_t = torch.tensor(IMAGENET_STD).view(3, 1, 1)

for idx in range(min(NUM_VIS, len(unet_train_ds))):
    img_tensor, mask_tensor = unet_train_ds[idx]
    img_np = (img_tensor * std_t + mean_t).permute(1, 2, 0).cpu().numpy()
    img_np = np.clip(img_np, 0, 1)
    mask_np = mask_tensor.squeeze().cpu().numpy()

    axes[0, idx].imshow(img_np)
    axes[0, idx].set_title(f"ROI crop {idx}")
    axes[0, idx].axis("off")

    axes[1, idx].imshow(mask_np, cmap="gray")
    axes[1, idx].set_title(f"GT mask {idx}")
    axes[1, idx].axis("off")

fig.suptitle("U-Net++ Training ROIs with Ground-Truth Masks", fontsize=13)
fig.tight_layout()
plt.show()

# ============================================================================
# 3.2 Build models
# ============================================================================

print("\nBuilding YOLO11m-seg model...")
yolo_model = build_yolo_model(CONFIG["yolo"].get("model", "yolo11m-seg.pt"))
print(f"  YOLO model loaded: {CONFIG['yolo'].get('model', 'yolo11m-seg.pt')}")

print("\nBuilding ROI segmentation model...")
unet_model = build_segmentation_model(CONFIG)
unet_model.to(device)
n_params = sum(p.numel() for p in unet_model.parameters())
print(f"  Segmentation model on {device} ({n_params:,} parameters)")
print(
    f"  Architecture: {CONFIG['unet'].get('architecture', 'unetplusplus')} | "
    f"Encoder: {CONFIG['unet']['encoder']} | Input: {CONFIG['unet']['input_size']}"
)

## 4: Training Loop / Validation / Checkpoints

In [ ]:
# ============================================================================
# 4.1 Stage 1 — YOLO11m-seg training
# ============================================================================

print("Starting Stage 1: YOLO11m-seg Training")
print("=" * 60)

yolo_results = train_yolo(CONFIG, SCRIPT_DIR)
yolo_test_metrics = evaluate_yolo(CONFIG, SCRIPT_DIR)
yolo_results.update(yolo_test_metrics)

print("\n" + "=" * 60)
print("YOLO Training Complete")
print("=" * 60)
for k, v in yolo_test_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

# ============================================================================
# 4.2 Stage 2 — ROI segmentation training
# ============================================================================

from experiment_io import get_unet_dirs

print("\n\nStarting Stage 2: ROI Segmentation Training")
print("=" * 60)

unet_results_summary = train_unet(CONFIG, SCRIPT_DIR)
best_dice = float(unet_results_summary.get("best_dice", 0.0))
best_epoch = int(unet_results_summary.get("best_epoch", 0))
unet_test_metrics = dict(unet_results_summary.get("test_metrics", {}))

results_unet = get_unet_dirs(SCRIPT_DIR, CONFIG)["results"]
with open(results_unet / "training_history.json", "r", encoding="utf-8") as f:
    unet_history = json.load(f)

print(f"\nBest Dice: {best_dice:.4f} at epoch {best_epoch}")
print(
    f"Training time: {unet_results_summary.get('training_time_s', 0):.0f}s "
    f"({unet_results_summary.get('training_time_s', 0)/60:.1f} min)"
)

In [ ]:
# ============================================================================
# Save training curves for both models
# ============================================================================

print("Saving U-Net++ training curves...")
save_unet_training_curves(unet_history, results_unet)

print("\nSaving YOLO prediction samples...")
predict_yolo(CONFIG, SCRIPT_DIR)

print("\nTraining curves were saved to disk.")
print("Use section 5 (Review) for the merged curve visualization.")

## 5: Qualitative Predictions / Reports / Summary & Review

In [ ]:
%matplotlib inline

import sys
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# ============================================================================
# Bootstrap — ensures this cell works even without running §1–§4 first
# ============================================================================

if "SCRIPT_DIR" not in globals():
    SCRIPT_DIR = Path.cwd().resolve()
if not (SCRIPT_DIR / "config.yaml").is_file() or not (SCRIPT_DIR / "train_model.py").is_file():
    raise RuntimeError(
        "Kernel cwd must be experiments/YOLO11m_UNetPP "
        f"(config.yaml + train_model.py not found in {SCRIPT_DIR})"
    )
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = SCRIPT_DIR.parent.parent

if "CONFIG" not in globals():
    from pipeline_utils import load_config, set_seed, get_device
    CONFIG = load_config(SCRIPT_DIR / "config.yaml", validate_combined=True)
    set_seed(CONFIG.get("seed", 42))
    device = get_device()
elif "device" not in globals():
    from pipeline_utils import get_device
    device = get_device()

if "data_root" not in globals() or "test_ann" not in globals():
    data_root = (PROJECT_ROOT / CONFIG["data_root"]).resolve()
    test_ann = (PROJECT_ROOT / CONFIG["ann_test"]).resolve()

if "results_dir" not in globals():
    results_dir = SCRIPT_DIR / "results"
    reports_dir = SCRIPT_DIR / "reports"
    for d in (results_dir, reports_dir):
        d.mkdir(parents=True, exist_ok=True)

from train_model import (
    build_yolo_model,
    predict_single_image,
    evaluate_combined,
    save_global_metrics_summary,
    generate_report,
    build_segmentation_model,
    load_unet_checkpoint,
    display_results_predictions,
)
from experiment_io import get_unet_dirs, get_combined_dirs, get_unet_best_checkpoint_path

if "unet_model" not in globals():
    unet_model = build_segmentation_model(CONFIG)
    unet_model.to(device)
    _ckpt = get_unet_best_checkpoint_path(SCRIPT_DIR, CONFIG)
    if _ckpt.is_file():
        load_unet_checkpoint(unet_model, _ckpt, device)
    else:
        print("[WARNING] No U-Net checkpoint found; using random weights.")

if "yolo_model" not in globals():
    yolo_model = build_yolo_model(CONFIG["yolo"].get("model", "yolo11m-seg.pt"))

if "yolo_results" not in globals():
    _yp = SCRIPT_DIR / "results" / "yolo" / "test_metrics.json"
    if _yp.is_file():
        with open(_yp, "r", encoding="utf-8") as f:
            yolo_results = json.load(f)
    else:
        yolo_results = {}

if "unet_results_summary" not in globals():
    _ud = get_unet_dirs(SCRIPT_DIR, CONFIG)["results"]
    _usp = _ud / "metrics_summary.json"
    if _usp.is_file():
        with open(_usp, "r", encoding="utf-8") as f:
            unet_results_summary = json.load(f)
    else:
        unet_results_summary = {"best_dice": 0.0, "best_epoch": 0, "test_metrics": {}}

if "best_dice" not in globals():
    best_dice = float(unet_results_summary.get("best_dice", 0.0))
if "best_epoch" not in globals():
    best_epoch = int(unet_results_summary.get("best_epoch", 0))
if "unet_test_metrics" not in globals():
    unet_test_metrics = dict(unet_results_summary.get("test_metrics", {}))

# ════════════════════════════════════════════════════════════════════════════
# PART A — Qualitative Predictions / Reports / Summary  (old §5)
# ════════════════════════════════════════════════════════════════════════════

# --- 5.1 Combined inference — single image example ---
yolo_best_path = SCRIPT_DIR / "checkpoints" / "yolo" / "best.pt"
if yolo_best_path.exists():
    yolo_model_best = build_yolo_model(str(yolo_best_path))
else:
    print("[WARNING] YOLO best.pt not found — using base model.")
    yolo_model_best = yolo_model

with open(str(test_ann), "r", encoding="utf-8") as f:
    test_coco = json.load(f)
test_images = test_coco["images"]

if test_images:
    sample_img = test_images[0]
    sample_path = str(data_root / sample_img["file_name"])

    overlay, info = predict_single_image(
        yolo_model_best, unet_model, sample_path, device, CONFIG,
    )
    print(f"File: {info['file_name']}")
    print(f"Wound area: {info.get('wound_area_cm2', 'N/A')} cm2 | "
          f"Infection: {info.get('infection', 'N/A')} | "
          f"Confidence: {info.get('confidence', 0):.2f}")

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Combined: {info.get('wound_area_cm2', '')} cm² | {info.get('infection', '')}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

# --- 5.2 Full combined evaluation on test set ---
print("\nRunning combined evaluation on full test set...")
combined_metrics = evaluate_combined(CONFIG, SCRIPT_DIR)

# --- 5.3 Save global metrics and generate report ---
save_global_metrics_summary(
    yolo_results, unet_results_summary, combined_metrics,
    CONFIG, results_dir,
)
generate_report(
    yolo_results, unet_results_summary, combined_metrics,
    CONFIG, reports_dir,
)

print("\n" + "=" * 80)
print("Training & Evaluation Summary")
print("=" * 80)
print(f"YOLO bbox mAP50:     {yolo_results.get('bbox_mAP50', 'N/A')}")
print(f"YOLO segm mAP50:     {yolo_results.get('segm_mAP50', 'N/A')}")
print(f"U-Net++ best Dice:   {best_dice:.4f} (epoch {best_epoch})")
print(f"U-Net++ test Dice:   {unet_test_metrics.get('dice', 'N/A')}")
print(f"Combined mean Dice:  {combined_metrics.get('mean_dice', 'N/A')}")
print(f"Combined mean IoU:   {combined_metrics.get('mean_iou', 'N/A')}")
print("=" * 80)

# ════════════════════════════════════════════════════════════════════════════
# PART B — Review: Display Saved Curves and Predictions  (old §6)
# ════════════════════════════════════════════════════════════════════════════

# --- Global Metrics Summary ---
print("\n" + "=" * 60)
print("GLOBAL METRICS SUMMARY")
print("=" * 60)

print("\n--- YOLO11m-seg ---")
for k in ["bbox_mAP50", "bbox_mAP50_95", "segm_mAP50", "segm_mAP50_95", "combined_AP50"]:
    print(f"  {k}: {yolo_results.get(k, 'N/A')}")

print("\n--- U-Net++ ---")
print(f"  Best Dice (val): {unet_results_summary.get('best_dice', 'N/A')}")
print(f"  Best epoch:      {unet_results_summary.get('best_epoch', 'N/A')}")
for k in ["dice", "iou", "pixel_accuracy"]:
    print(f"  Test {k}: {unet_test_metrics.get(k, 'N/A')}")

print("\n--- Combined Pipeline ---")
comb_dirs = get_combined_dirs(SCRIPT_DIR, CONFIG)
comb_metrics_path = comb_dirs["results"] / "metrics_summary.json"
if comb_metrics_path.exists():
    with open(comb_metrics_path, "r", encoding="utf-8") as f:
        _comb_metrics = json.load(f)
    for k in ["mean_dice", "mean_iou", "n_images_evaluated",
              "coco_bbox_AP50", "coco_bbox_AP75",
              "coco_segm_AP50", "coco_segm_AP75",
              "coco_combined_AP50", "coco_combined_AP75"]:
        print(f"  {k}: {_comb_metrics.get(k, 'N/A')}")
else:
    print("  (metrics_summary.json not found — run combined evaluation first)")

# --- Merged Training Curves ---
print("\n" + "=" * 60)
print("TRAINING CURVES")
print("=" * 60)

unet_dirs = get_unet_dirs(SCRIPT_DIR, CONFIG)
yolo_csv_path = SCRIPT_DIR / "checkpoints" / "yolo" / "train" / "results.csv"
unet_json_path = unet_dirs["results"] / "training_history.json"

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

if yolo_csv_path.exists():
    df = pd.read_csv(yolo_csv_path)
    df.columns = [c.strip() for c in df.columns]
    epoch_col = "epoch" if "epoch" in df.columns else df.columns[0]

    ax = axes[0, 0]
    for col, label in [
        ("metrics/mAP50(B)", "bbox mAP50"),
        ("metrics/mAP50-95(B)", "bbox mAP50-95"),
        ("metrics/mAP50(M)", "segm mAP50"),
        ("metrics/mAP50-95(M)", "segm mAP50-95"),
    ]:
        if col in df.columns:
            ax.plot(df[epoch_col], df[col], label=label)
    ax.set_title("YOLO Metrics"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    for col, label in [
        ("train/box_loss", "box loss"), ("train/seg_loss", "seg loss"),
        ("train/cls_loss", "cls loss"), ("train/dfl_loss", "dfl loss"),
    ]:
        if col in df.columns:
            ax.plot(df[epoch_col], df[col], label=label)
    ax.set_title("YOLO Losses"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
else:
    axes[0, 0].text(0.5, 0.5, "YOLO results.csv not found", ha="center", va="center")
    axes[0, 1].text(0.5, 0.5, "YOLO results.csv not found", ha="center", va="center")

if unet_json_path.exists():
    with open(unet_json_path, "r", encoding="utf-8") as f:
        hist = json.load(f)
    _tl = hist.get("train_losses") or hist.get("train_loss") or []
    _vl = hist.get("val_losses") or hist.get("val_loss") or []
    _vd = hist.get("dice_per_epoch") or hist.get("val_dice") or []
    _vi = hist.get("iou_per_epoch") or hist.get("val_iou") or []
    epochs_list = list(range(1, len(_tl) + 1))

    ax = axes[1, 0]
    if _tl:
        ax.plot(epochs_list, _tl, label="train loss")
    if _vl:
        ax.plot(epochs_list, _vl, label="val loss")
    ax.set_title("U-Net Losses"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    if _vd:
        ax.plot(epochs_list, _vd, label="val Dice")
    if _vi:
        ax.plot(epochs_list, _vi, label="val IoU")
    ax.set_title("U-Net Metrics"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
else:
    axes[1, 0].text(0.5, 0.5, "U-Net history not found", ha="center", va="center")
    axes[1, 1].text(0.5, 0.5, "U-Net history not found", ha="center", va="center")

fig.suptitle("Training Curves — YOLO + U-Net++", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

# --- Qualitative Predictions ---
print("\n" + "=" * 60)
print("QUALITATIVE PREDICTIONS")
print("=" * 60)
display_results_predictions(SCRIPT_DIR / "results", n_show=8)

# --- Combined Pipeline COCO Markdown Report ---
if comb_metrics_path.exists():
    with open(comb_metrics_path, "r", encoding="utf-8") as f:
        _m = json.load(f)
    md_lines = ["---", "", "## Combined Pipeline Results", "",
                "| Metric | Value |", "|--------|-------|"]
    for k, v in _m.items():
        md_lines.append(f"| {k} | {v} |")
    md_lines += ["", "---", "", "## Infection Classification Results", "",
                 "*Not available.*"]
    display(Markdown("\n".join(md_lines)))

# --- Checkpoints ---
print("\n" + "=" * 60)
print("CHECKPOINTS")
print("=" * 60)
ckpt_base = SCRIPT_DIR / "checkpoints"
for sub in ["yolo", "unet"]:
    sub_dir = ckpt_base / sub
    if not sub_dir.exists():
        continue
    print(f"\n  {sub}/")
    for f in sorted(sub_dir.glob("*.pt")) + sorted(sub_dir.glob("*.pth")):
        sz_mb = f.stat().st_size / (1024 * 1024)
        print(f"    {f.name} ({sz_mb:.1f} MB)")